#### Import and data ingestion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
df = pd.read_csv('../data/processed/dataset_with_nlp_features.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
print(f"Loaded {df.shape[0]} trading days, {df.shape[1]} features.")

#### Target distribution and macro volatility exploration

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# JISDOR Exchange Rate Level
axes[0, 0].plot(df['date'], df['jisdor_rate'], color='#1f77b4', lw=1.5)
axes[0, 0].set_title('USD/IDR JISDOR Exchange Rate (2021–2026)')
axes[0, 0].set_ylabel('IDR per USD')

# Return Distribution
sns.histplot(df['pct_change'].dropna(), kde=True, ax=axes[0, 1], color='#2ca02c')
axes[0, 1].set_title('Distribution of Daily Percentage Change')
axes[0, 1].set_xlabel('Percentage Return')

# Direction Target Count
sns.countplot(x='direction', data=df.dropna(subset=['direction']), ax=axes[1, 0], palette='Blues_r')
axes[1, 0].set_title('Direction Target Imbalance (+1: USD Up, -1: IDR Up, 0: Flat)')
axes[1, 0].set_xticklabels(['IDR Gain (-1)', 'Flat (0)', 'USD Gain (+1)'])

# Correlation of NLP Sentiment vs. Returns
corr_cols = ['pct_change', 'lex_net_polarity', 'lex_unc_count', 'ent_usd', 'ent_fed', 'ent_geopolitics']
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.3f', cmap='coolwarm', ax=axes[1, 1])
axes[1, 1].set_title('Correlation Heatmap: Sentiment & Macro Returns')

plt.tight_layout()
plt.show()

#### Model benchmark performance visualisation

In [ ]:
import json
with open('../data/processed/model_benchmark_results.json') as f:
    res = json.load(f)
res_df = pd.DataFrame(res)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(res_df['model'], res_df['test_accuracy'] * 100, color='#3470a3')
ax.axvline(57.08, color='crimson', linestyle='--', label='Majority Naive Baseline (57.08%)')
ax.set_xlabel('Out-of-Sample Test Accuracy (%)')
ax.set_title('Model Performance Comparison: Pure Time-Series vs Combined NLP')
ax.set_xlim(40, 65)
for bar in bars:
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f"{bar.get_width():.2f}%", va='center')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()